<a href="https://colab.research.google.com/github/ghinaiyariken/Fly_Rank_Internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Search Ranking & Discoverability Capstone
## Refresh / Content Opportunity Scoring

**Research question:** Can observable search-performance and content-age signals help prioritize which content items deserve human review for refresh, expansion, protection, pruning, or monitoring?

**Decision supported:** Which content items should an SEO/content lead inspect first when review capacity is limited?

**Output:** A reproducible ranked review queue with scores, actions, reason codes, confidence notes, and an honest comparison of a transparent baseline against learned models.

This notebook is the capstone continuation of the Week 1–4 work: research question → data contract → transparent baseline → learned model → validation → action recommendations.

> **Security:** Never paste a Hugging Face token into a cell. In Google Colab, store the READ token in **Secrets** under `HF_TOKEN` and let the notebook retrieve it at runtime.

## 1. Question

The capstone uses **Lane 2 — Refresh / Content Opportunity Scoring**.

The project does **not** attempt to predict Google's ranking algorithm or prove that refreshing a page causes recovery. It is a decision-support system: it ranks items for human review using information available before a future outcome window.

The practical question is: **given limited review capacity, can a learned ranking improve on a simple, transparent rule for finding content items that later show an observed search-performance decline?**

In [1]:
# Basic environment and secure Hugging Face access.
%pip -q install duckdb scikit-learn pandas numpy matplotlib huggingface_hub

import os
import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. In Colab, click the Secrets key icon, add HF_TOKEN, "
        "paste your Hugging Face READ token there, and enable notebook access."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
os.environ["HF_TOKEN"] = HF_TOKEN
con.execute("CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, PROVIDER credential_chain)")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM = f"{REL}/dim_content.parquet"

print("Authenticated through the protected Colab Secret HF_TOKEN.")
print("No token value is printed or stored in this notebook.")

Authenticated through the protected Colab Secret HF_TOKEN.
No token value is printed or stored in this notebook.


## 2. Data

**Source:** `FlyRank/internship-warehouse`, build `flyrank_pseudonymized_warehouse_release_v20260703`.

**Tables used:**
- `fact_content_daily_performance` — daily GSC/search-performance facts.
- `dim_content` — content metadata used for content age and safe joins.

**Unit of analysis:** one pseudonymized `client_hash_id × content_hash_id` at the end of a monthly feature window.

**Feature windows:** monthly aggregates from February through May 2026.

**Outcome window:** the immediately following month. For a feature month `M`, the label is whether next-month impressions are at least 20% lower than month `M`, among items measured in both windows.

**Held-out evaluation:** May 2026 features → June 2026 outcome. June is treated as the final test month and is not used to choose features, thresholds, or model settings.

**Excluded:** `trend_direction`, `trend_pct`, product decision fields such as `health_score`, `priority_score`, `action_type`, and any future-window values. Pseudonymous IDs are context/join/split fields, never model features. No client names, domains, URLs, private queries, credentials, or raw exports are published.

In [3]:
import duckdb

# Warehouse sanity check: source date coverage and grain.
month_path = f'{FACT}/month=2026-03/*.parquet'
source_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_daily_keys,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{month_path}')
""").df()
display(source_check)
assert source_check.loc[0, "rows"] == source_check.loc[0, "distinct_daily_keys"]
print("PASS: daily fact grain matches client × content × report_date.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,distinct_daily_keys,min_date,max_date
0,9841378,9841378,2026-03-01,2026-03-31


PASS: daily fact grain matches client × content × report_date.


### Data availability rule

The warehouse documentation warns that GSC availability is not the same thing as a zero metric. The notebook therefore uses `gsc_data_available IS TRUE` when aggregating search measures. Rows without measured GSC data are not silently treated as zero performance.

In [6]:
def monthly_features(month):
    path = f"{FACT}/month={month}/*.parquet"
    q = f"""
    WITH agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions,
            SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS clicks,
            SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE)
              / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE), 0) AS avg_position,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_measured_days
        FROM read_parquet('{path}')
        GROUP BY 1, 2
    )
    SELECT
        a.client_hash_id,
        a.content_hash_id,
        a.impressions,
        a.clicks,
        a.avg_position,
        a.gsc_measured_days,
        LAST_DAY(DATE '{month}-01') - CAST(c.content_created_date AS DATE) AS content_age_days
    FROM agg a
    LEFT JOIN read_parquet('{DIM}') c USING (content_hash_id)
    WHERE a.gsc_measured_days > 0
      AND c.content_created_date IS NOT NULL
    """
    df = con.sql(q).df()
    # Only the feature-month information is used here.
    df["ctr"] = np.where(df["impressions"] > 0, df["clicks"] / df["impressions"], np.nan)
    df["month"] = month
    return df

def add_outcome(feature_df, next_month):
    path = f"{FACT}/month={next_month}/*.parquet"
    future = con.sql(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS next_impressions,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS next_measured_days
        FROM read_parquet('{path}')
        GROUP BY 1, 2
    """).df()
    out = feature_df.merge(future, on=["client_hash_id", "content_hash_id"], how="left")
    valid = (
        out["impressions"].gt(0)
        & out["next_impressions"].notna()
        & out["next_measured_days"].gt(0)
    )
    out["declined_next_month"] = np.where(
        valid,
        (out["next_impressions"] < 0.80 * out["impressions"]).astype(int),
        np.nan,
    )
    return out

# Development windows. Each sample uses only information available at its feature-month end.
month_pairs = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05")]
train_parts = []
for m, nxt in month_pairs:
    try:
        x = add_outcome(monthly_features(m), nxt)
        x["feature_month"] = m
        train_parts.append(x)
        print(m, "→", nxt, "rows:", f"{len(x):,}", "labeled:", f"{x['declined_next_month'].notna().sum():,}")
    except Exception as e:
        print(f"Error processing month pair {m} → {nxt}: {e}")

dev = pd.concat(train_parts, ignore_index=True)
print("Development rows:", f"{len(dev):,}")
print("Development decline base rate:", f"{dev['declined_next_month'].mean():.3f}")

2026-02 → 2026-03 rows: 153,559 labeled: 134,238


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-03 → 2026-04 rows: 176,738 labeled: 158,549


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04 → 2026-05 rows: 194,760 labeled: 183,345
Development rows: 525,057
Development decline base rate: 0.397


## 3. Methodology

### Label

`declined_next_month = 1` means next-month GSC impressions are **more than 20% below** the feature-month impressions, with measurable GSC data in both windows.

This is an **observed future outcome**, not a claim that content refresh causes recovery.

### Five model features

1. `impressions` — feature-month search visibility.
2. `clicks` — feature-month clicks.
3. `avg_position` — feature-month average position, weighted by impressions.
4. `gsc_measured_days` — number of days with measured GSC data.
5. `content_age_days` — age at the feature-month decision point.

The model does **not** use `trend_direction`, `trend_pct`, future impressions, future clicks, labels, product scores, or pseudonymous IDs.

### Baseline

The Week-4 transparent rule is adapted to the warehouse:

> **Review first when a content item is at least 180 days old and has at least 500 feature-month impressions.**

This is deliberately simple and explainable. The learned model must beat it on the same held-out population and metric to earn its complexity.

### Validation

The final evaluation is **time-aware**: development uses earlier month→next-month pairs; the final held-out test uses May features to predict June. This mirrors deployment and avoids random mixing of future observations into training.

In [7]:
FEATURES = [
    "impressions",
    "clicks",
    "avg_position",
    "gsc_measured_days",
    "content_age_days",
]

# Explicit feature audit.
forbidden = {
    "trend_direction", "trend_pct", "declined_next_month",
    "next_impressions", "next_measured_days", "health_score",
    "priority_score", "action_type", "client_hash_id", "content_hash_id"
}
assert set(FEATURES).isdisjoint(forbidden)
print("Final model features:", FEATURES)
print("PASS: no label/future/product-decision/ID field is in the feature set.")

Final model features: ['impressions', 'clicks', 'avg_position', 'gsc_measured_days', 'content_age_days']
PASS: no label/future/product-decision/ID field is in the feature set.


### Leakage attack

A deliberately invalid feature is constructed from the future outcome window. It should look artificially strong. It is then removed and is not used in the final model.

In [8]:
from sklearn.metrics import roc_auc_score

# Demonstration only: future change is directly derived from the outcome period.
leak_demo = dev[dev["declined_next_month"].notna()].copy()
leak_demo["LEAK_future_change"] = (
    leak_demo["next_impressions"] - leak_demo["impressions"]
) / leak_demo["impressions"].replace(0, np.nan)
leak_auc = roc_auc_score(leak_demo["declined_next_month"], -leak_demo["LEAK_future_change"])
print(f"Invalid future-leak AUC: {leak_auc:.3f}")
print("This number is intentionally invalid because the feature reads the future outcome window.")

leak_demo = leak_demo.drop(columns=["LEAK_future_change"])
assert "LEAK_future_change" not in FEATURES
print("PASS: leaked field removed before modeling.")

Invalid future-leak AUC: 1.000
This number is intentionally invalid because the feature reads the future outcome window.
PASS: leaked field removed before modeling.


## 4. Results (vs baseline)

The primary ranking metric is **Precision@20**, because the Week-2 framing defined the practical unit of review capacity as a top-20 queue. The table also reports Precision@50 and the overall decline base rate.

AUC is included as a secondary discrimination metric. The baseline and model are evaluated on the **same held-out population**.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

train = dev[dev["declined_next_month"].notna()].copy()

logit = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=50,
        class_weight="balanced", random_state=42, n_jobs=-1
    )),
])

logit.fit(train[FEATURES], train["declined_next_month"].astype(int))
rf.fit(train[FEATURES], train["declined_next_month"].astype(int))

# Held-out May -> June.
test = add_outcome(monthly_features("2026-05"), "2026-06")
test = test[test["declined_next_month"].notna()].copy()
test["baseline_score"] = ((test["content_age_days"] >= 180) & (test["impressions"] >= 500)).astype(int)
test["logit_score"] = logit.predict_proba(test[FEATURES])[:, 1]
test["rf_score"] = rf.predict_proba(test[FEATURES])[:, 1]

def p_at_k(df, score_col, k):
    return df.sort_values(score_col, ascending=False).head(min(k, len(df)))["declined_next_month"].mean()

def result_row(name, score_col):
    return {
        "method": name,
        "Precision@20": p_at_k(test, score_col, 20),
        "Precision@50": p_at_k(test, score_col, 50),
        "AUC": roc_auc_score(test["declined_next_month"], test[score_col]),
        "base_rate": test["declined_next_month"].mean(),
    }

results = pd.DataFrame([
    result_row("Week-4 transparent baseline", "baseline_score"),
    result_row("Logistic regression", "logit_score"),
    result_row("Random forest", "rf_score"),
])

display(results)
print("Held-out rows:", f"{len(test):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,method,Precision@20,Precision@50,AUC,base_rate
0,Week-4 transparent baseline,0.8,0.78,0.519661,0.647187
1,Logistic regression,0.8,0.82,0.572406,0.647187
2,Random forest,0.8,0.60,0.597144,0.647187


Held-out rows: 184,877


### Error analysis

The model is a prioritization aid. A false positive means an item is ranked highly but does not meet the observed decline label; a false negative means a later decline is ranked below the chosen queue cut.

The review below intentionally shows concrete pseudonymous cases and explains why they are difficult. The identifiers are hashes only and are not model features.

In [10]:
top20 = test.sort_values("rf_score", ascending=False).head(20).copy()
top20["is_error"] = top20["declined_next_month"].eq(0)
errors = top20[top20["is_error"]].head(3).copy()

error_view = errors[[
    "client_hash_id", "content_hash_id", "rf_score",
    "impressions", "clicks", "avg_position", "content_age_days",
    "declined_next_month"
]].copy()
error_view["why_hard"] = (
    "High observed signal can still be followed by stability or recovery; "
    "seasonality, query mix, SERP changes, or measurement variation are not fully represented."
)
display(error_view)

# Simple permutation importance on the held-out feature frame for interpretation.
from sklearn.inspection import permutation_importance
perm = permutation_importance(
    rf, test[FEATURES], test["declined_next_month"].astype(int),
    scoring="roc_auc", n_repeats=3, random_state=42, n_jobs=-1
)
importance = pd.DataFrame({"feature": FEATURES, "mean_auc_drop": perm.importances_mean})    .sort_values("mean_auc_drop", ascending=False)
display(importance)
print("Interpretation warning: importance is association within this evaluation design, not causality.")

,client_hash_id,content_hash_id,rf_score,impressions,clicks,avg_position,content_age_days,declined_next_month,why_hard
107390,client_23a62021009f63c4,content_0a2272d17ba57b76,0.806229,724.0,1.0,1.389503,86,0.0,High observed signal can still be followed by ...
46319,client_62f4a7e64f5e0096,content_5cd8d3e25b025e74,0.793497,482.0,1.0,1.995851,86,0.0,High observed signal can still be followed by ...
218447,client_0fa64a184f18a4a0,content_612f4efcfecbe97e,0.793424,1390.0,1.0,2.125899,96,0.0,High observed signal can still be followed by ...


,feature,mean_auc_drop
3,gsc_measured_days,0.029057
1,clicks,0.024359
0,impressions,0.016204
4,content_age_days,0.013378
2,avg_position,0.002507


Interpretation warning: importance is association within this evaluation design, not causality.


## 5. Limitations & honest framing

- The target is an **observed next-month impression decline**, not proof that a refresh would help.
- Search performance can change because of seasonality, query mix, SERP changes, technical issues, or measurement changes.
- The model is evaluated on a temporal holdout; performance may change on later months or different clients.
- The warehouse is an unbalanced panel, so history depth differs across clients.
- Content age is a proxy for freshness, not evidence that the content is outdated.
- Recommendations are for **human review**, not automatic editing, pruning, or publication.
- This work does not predict Google's ranking algorithm and does not establish causal impact.

In [11]:
# Final leakage and safety assertions.
assert set(FEATURES).isdisjoint(forbidden)
assert "LEAK_future_change" not in FEATURES
assert "declined_next_month" not in FEATURES
assert "next_impressions" not in FEATURES
print("PASS: final feature set is free of known future/label/product-decision/ID leakage.")
print("PASS: final test is May 2026 feature window -> June 2026 outcome window.")

PASS: final feature set is free of known future/label/product-decision/ID leakage.
PASS: final test is May 2026 feature window -> June 2026 outcome window.


## 6. Ranked recommendations

The final action queue is ranked by the selected model. Reason codes use only observable feature-window information. A recommendation means **inspect first**, not **change automatically**.

In [12]:
queue = test.copy()
queue["score"] = queue["rf_score"]


def reason_code(r):
    if r["content_age_days"] >= 180 and r["impressions"] >= 500:
        return "old_and_visible"
    if r["impressions"] >= 500 and r["avg_position"] <= 10:
        return "visible_page_one"
    if r["impressions"] >= 500:
        return "visible_search_demand"
    return "lower_observed_volume"

def action_from_reason(reason):
    return {
        "old_and_visible": "review_refresh",
        "visible_page_one": "review_content_and_intent",
        "visible_search_demand": "inspect_search_opportunity",
        "lower_observed_volume": "monitor",
    }[reason]

queue["reason_code"] = queue.apply(reason_code, axis=1)
queue["action"] = queue["reason_code"].map(action_from_reason)
queue = queue.sort_values(["score", "impressions"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

# Public-safe output: only pseudonymous IDs and measured metrics.
public_cols = [
    "rank", "client_hash_id", "content_hash_id", "score",
    "reason_code", "action", "impressions", "clicks",
    "avg_position", "gsc_measured_days", "content_age_days"
]
public_queue = queue[public_cols].head(100).copy()
display(public_queue.head(20))

,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions,clicks,avg_position,gsc_measured_days,content_age_days
0,1,client_23a62021009f63c4,content_0a2272d17ba57b76,0.806229,visible_page_one,review_content_and_intent,724.0,1.0,1.389503,30,86
1,2,client_20259bd6705d81d4,content_d7aa781756a47b5f,0.805974,old_and_visible,review_refresh,904.0,1.0,0.956858,30,215
2,3,client_810019792c9b8efc,content_783ef12fb7501fbc,0.802667,visible_page_one,review_content_and_intent,1243.0,1.0,2.032985,30,61
3,4,client_62f4a7e64f5e0096,content_b3df4c79f98d291b,0.800311,visible_page_one,review_content_and_intent,4308.0,0.0,0.380223,30,171
4,5,client_a80fca3f171ed1de,content_1b2092631f0fdb05,0.799049,lower_observed_volume,monitor,438.0,0.0,1.716895,30,108
5,6,client_23a62021009f63c4,content_ccab5c41c5bd735c,0.798452,visible_page_one,review_content_and_intent,1282.0,0.0,2.394696,30,115
6,7,client_0fa64a184f18a4a0,content_57a642bc5429d1bd,0.796993,visible_page_one,review_content_and_intent,791.0,1.0,1.508217,31,96
7,8,client_e5c2aa26a8598242,content_cf2899a946970068,0.794722,visible_page_one,review_content_and_intent,2321.0,2.0,1.472641,31,59
8,9,client_157ffe4d4a595515,content_194f0922b3a5343b,0.793928,visible_page_one,review_content_and_intent,2957.0,1.0,0.986473,31,45
9,10,client_62f4a7e64f5e0096,content_5cd8d3e25b025e74,0.793497,lower_observed_volume,monitor,482.0,1.0,1.995851,31,86


In [13]:
# Human-review notes for the top 20.
top20_review = public_queue.head(20).copy()
top20_review["confidence"] = np.select(
    [top20_review["impressions"] >= 3000, top20_review["impressions"] >= 500],
    ["higher observed volume", "moderate observed volume"],
    default="lower observed volume"
)
top20_review["what_would_make_it_wrong"] = (
    "Seasonality, query/SERP changes, technical issues, or measurement changes could explain the pattern."
)
display(top20_review)

,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions,clicks,avg_position,gsc_measured_days,content_age_days,confidence,what_would_make_it_wrong
0,1,client_23a62021009f63c4,content_0a2272d17ba57b76,0.806229,visible_page_one,review_content_and_intent,724.0,1.0,1.389503,30,86,moderate observed volume,"Seasonality, query/SERP changes, technical iss..."
1,2,client_20259bd6705d81d4,content_d7aa781756a47b5f,0.805974,old_and_visible,review_refresh,904.0,1.0,0.956858,30,215,moderate observed volume,"Seasonality, query/SERP changes, technical iss..."
2,3,client_810019792c9b8efc,content_783ef12fb7501fbc,0.802667,visible_page_one,review_content_and_intent,1243.0,1.0,2.032985,30,61,moderate observed volume,"Seasonality, query/SERP changes, technical iss..."
3,4,client_62f4a7e64f5e0096,content_b3df4c79f98d291b,0.800311,visible_page_one,review_content_and_intent,4308.0,0.0,0.380223,30,171,higher observed volume,"Seasonality, query/SERP changes, technical iss..."
4,5,client_a80fca3f171ed1de,content_1b2092631f0fdb05,0.799049,lower_observed_volume,monitor,438.0,0.0,1.716895,30,108,lower observed volume,"Seasonality, query/SERP changes, technical iss..."
5,6,client_23a62021009f63c4,content_ccab5c41c5bd735c,0.798452,visible_page_one,review_content_and_intent,1282.0,0.0,2.394696,30,115,moderate observed volume,"Seasonality, query/SERP changes, technical iss..."
6,7,client_0fa64a184f18a4a0,content_57a642bc5429d1bd,0.796993,visible_page_one,review_content_and_intent,791.0,1.0,1.508217,31,96,moderate observed volume,"Seasonality, query/SERP changes, technical iss..."
7,8,client_e5c2aa26a8598242,content_cf2899a946970068,0.794722,visible_page_one,review_content_and_intent,2321.0,2.0,1.472641,31,59,moderate observed volume,"Seasonality, query/SERP changes, technical iss..."
8,9,client_157ffe4d4a595515,content_194f0922b3a5343b,0.793928,visible_page_one,review_content_and_intent,2957.0,1.0,0.986473,31,45,moderate observed volume,"Seasonality, query/SERP changes, technical iss..."
9,10,client_62f4a7e64f5e0096,content_5cd8d3e25b025e74,0.793497,lower_observed_volume,monitor,482.0,1.0,1.995851,31,86,lower observed volume,"Seasonality, query/SERP changes, technical iss..."


## 7. Artifacts the paper embeds

The next cell writes the reproducibility artifacts used by the research paper. It does not publish client names, domains, URLs, private queries, credentials, or raw warehouse exports.

In [14]:
from pathlib import Path
Path("work/outputs").mkdir(parents=True, exist_ok=True)

public_queue.to_csv("work/outputs/capstone_ranked_recommendations.csv", index=False)
results.to_csv("work/outputs/capstone_model_vs_baseline.csv", index=False)
importance.to_csv("work/outputs/capstone_feature_importance.csv", index=False)

# A compact metrics receipt for the final held-out run.
receipt = {
    "test_window": "May 2026 features -> June 2026 outcome",
    "test_rows": int(len(test)),
    "decline_base_rate": float(test["declined_next_month"].mean()),
    "baseline_precision_at_20": float(p_at_k(test, "baseline_score", 20)),
    "logistic_precision_at_20": float(p_at_k(test, "logit_score", 20)),
    "random_forest_precision_at_20": float(p_at_k(test, "rf_score", 20)),
    "baseline_precision_at_50": float(p_at_k(test, "baseline_score", 50)),
    "logistic_precision_at_50": float(p_at_k(test, "logit_score", 50)),
    "random_forest_precision_at_50": float(p_at_k(test, "rf_score", 50)),
}
pd.DataFrame([receipt]).to_json("work/outputs/capstone_metrics_receipt.json", orient="records", indent=2)

print("Wrote:")
for p in [
    "work/outputs/capstone_ranked_recommendations.csv",
    "work/outputs/capstone_model_vs_baseline.csv",
    "work/outputs/capstone_feature_importance.csv",
    "work/outputs/capstone_metrics_receipt.json",
]:
    print(" -", p)

Wrote:
 - work/outputs/capstone_ranked_recommendations.csv
 - work/outputs/capstone_model_vs_baseline.csv
 - work/outputs/capstone_feature_importance.csv
 - work/outputs/capstone_metrics_receipt.json


## Paper-ready interpretation

Use the actual executed values from the output tables in the public paper. Do **not** replace them with estimates or invented numbers.

The core claim should follow this pattern:

> **Observed:** On the held-out May→June window, the selected model achieved [actual Precision@20], compared with [actual baseline Precision@20] for the transparent rule, against a [actual base rate] decline rate.

> **Directional:** This suggests the learned ranking may improve prioritization under this evaluation design.

> **Not proven:** The analysis does not show that refreshing a recommended item causes recovery, and it does not predict Google's ranking algorithm.

## Reproducibility

Environment: Python + DuckDB + pandas + scikit-learn. Random seed: **42** for the learned models and permutation-importance analysis.

To reproduce:

1. Clone the repository.
2. Open this notebook in Google Colab.
3. Add your Hugging Face READ token as a Colab Secret named `HF_TOKEN`.
4. Run all cells.
5. Confirm the final held-out metrics and output files are regenerated.
6. Commit the executed notebook and generated public-safe artifacts.

The token must never be committed to GitHub.

## 8. Acknowledgments & data credit

Built on the **FlyRank ML Internship dataset**. Data source: FlyRank — https://flyrank.ai

The analysis is presented as decision-support research using a pseudonymized warehouse release. It does not expose client-identifying information or claim causal effects.

## ML-12 closing deliverables

### 5-minute demo outline
1. **Question (30s):** Which content items deserve review first?
2. **Data (45s):** Monthly search-performance facts + content age from the FlyRank warehouse.
3. **Baseline (45s):** Simple old + visible rule.
4. **Model (60s):** Logistic regression and random forest compared on the same time-aware holdout.
5. **Leakage check (45s):** Deliberate future-feature test, then removal.
6. **Results (45s):** Precision@20/50 and base rate.
7. **Recommendations (30s):** Ranked queue with reason codes and human-review limits.

### Social-post cut
> I built a decision-support ranking system on the FlyRank ML Internship warehouse to prioritize content-review opportunities. I compared a transparent “old + visible” baseline with learned models using a time-aware future-month test and an explicit leakage audit. The result is a reproducible ranked queue with reason codes—not a claim about Google’s algorithm or a guarantee that refreshing a page will recover traffic.

### Employer-facing 3 sentences
> I built a reproducible ML ranking pipeline that prioritizes content-review candidates from real search-performance data. I designed the data contract, time-aware validation, transparent baseline, learned models, leakage audit, and public-safe recommendation queue end to end. The project demonstrates practical ML judgment: matching the model to the decision, comparing against a baseline, and separating observed evidence from causal claims.

## 9. Final self-check

- [ ] Correct lane: Refresh / Content Opportunity Scoring.
- [ ] Research question and decision are explicit.
- [ ] Full FlyRank warehouse is used for capstone execution; `_sample` is not used for development.
- [ ] Feature windows precede the outcome window.
- [ ] Five model features are listed and verified.
- [ ] Baseline and model use the same held-out population and metrics.
- [ ] Base rate is reported next to ranking metrics.
- [ ] Deliberate leakage experiment is shown and leaked field is removed.
- [ ] Error analysis includes concrete wrong cases.
- [ ] Ranked recommendations include reason codes and human-review limits.
- [ ] No client names, domains, URLs, private queries, credentials, or raw exports appear in public outputs.
- [ ] Paper has all required sections, including Abstract and Acknowledgments/data credit.
- [ ] `submission/paper_url.txt` contains exactly one public paper URL.
- [ ] Executed notebook and outputs are committed to the repository.
- [ ] Only the repository URL is submitted on the FlyRank capstone card.